> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `pipelinetools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `PIPELINETOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `PIPELINETOOLS`.

This module is particularly valuable for researchers managing large-scale processing pipelines: selecting
the subject IDs to process, building file inventories of the BIDS derivatives folders, and assessing,
storing and querying the processing status of the subjects, either as JSON files or as SQLite3
databases.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`Utility Methods`](#pipe-utilities). Utility methods to select the subjects to process
    * 🔸 1.1. [`get_fullids2process`](#pipe-get_fullids2process). Get the list of Full IDs to process from various input sources

* 🔷 2. [`Inventory of the Derivatives`](#pipe-inventory). Methods to scan the derivatives folders and build the file inventories
    * 🔸 2.1. [`build_derivatives_inventory`](#pipe-build_derivatives_inventory). Build a combined file inventory across all the pipeline derivative folders

* 🔷 3. [`Processing Status of the Pipelines`](#pipe-status). Methods for assessing the processing status of the pipelines
    * 🔸 3.1. [`create_processing_status_table`](#pipe-create_processing_status_table). Create a table with the processing status of the subjects in the BIDS derivatives directory
    * 🔸 3.2. [`get_processing_status_details_json`](#pipe-get_processing_status_details_json). Create a dictionary with the details of the processing status of the subjects and store it as a JSON file
    * 🔸 3.3. [`get_processing_status_details_sqlite3`](#pipe-get_processing_status_details_sqlite3). Create a dictionary with the details of the processing status of the subjects and store it in a SQLite3 database
    * 🔸 3.4. [`query_processing_status_db`](#pipe-query_processing_status_db). Query the processing status database to extract useful information

---

---
## <a id="pipe-utilities"></a>1. Utility Methods
Utility methods to select the subjects to process.

### 1.1 `get_fullids2process`
<a id="pipe-get_fullids2process"></a>

Get the list of Full IDs to process from various input sources.

In [ ]:
######################## Testing get_fullids2process ########################
import tempfile
from pathlib import Path
import clabtoolkit.pipelinetools as cltpipe
import clabtoolkit.bidstools as cltbids

print('Testing get_fullids2process function with different inputs')

# Generating a BIDs dataset
bids_dir = str(Path(tempfile.gettempdir()) / "sim_bids_test0")   # <-- Replace with your BIDS root directory

# Create a simulated BIDS dataset for testing purposes
print(f'Test 1: Creating a simulated BIDS dataset in {bids_dir}...')

nsub = 3 # Number of subjects
nses = 2 # Number of visits
nruns = 2 # Number of runs per modality
modalities = ["anat", "dwi", "func"] # Define the modalities to include in the simulated dataset
tasks = ["rest", "nback"] # Define the tasks for the fMRI images that will be included in the simulated dataset

print(f'Number of subjects: {nsub}')
print(f'Number of visits: {nses}')

cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=nsub,
        n_visits=nses,
        modalities=modalities,
        tasks=tasks,
        include_derivatives=None,
        runs_per_modality=nruns,
        overwrite=True,
    )

## Printing BIDS dataset structure
print(cltbids.generate_bids_tree(bids_dir))

# Test 1: No IDs provided (discover from directory). It will automatically find all the T1w files.
print('\nTest 1: No IDs provided')
print(cltpipe.get_fullids2process(in_dir=str(Path(tempfile.gettempdir()) / 'sim_bids_test0'), source = "files"))

# Test 2: Discover subjects from files (files)
print('\nTest 2: Discover subjects from files (files) but selecting only dwi files')
print(cltpipe.get_fullids2process(in_dir=str(Path(tempfile.gettempdir()) / 'sim_bids_test0'), source = "files", or_filter = "dwi"))

# Test 3: Discover subjects from directory (files)
print('\nTest 3: Discover subjects from directory (folders)')
print(cltpipe.get_fullids2process(in_dir=str(Path(tempfile.gettempdir()) / 'sim_bids_test0'), source = "folders"))

# Test 4: List of IDs
print('\nTest 4: List of IDs')
print(cltpipe.get_fullids2process(['sub-001', 'sub-002']))

# Test 5: Comma-separated string
print('\nTest 5: Comma-separated string')
print(cltpipe.get_fullids2process('sub-001_ses-V1, sub-002_ses-V1, sub-003_ses-V1_acq-mprage'))

# Test 6: Single ID
print('\nTest 6: Single ID')
print(cltpipe.get_fullids2process('sub-001'))

# Test 7: Text file
print('\nTest 7: Text file')

# Create a text file with subject IDs for testing
with open(str(Path(tempfile.gettempdir()) / 'all_t1s_ids.txt'), 'w') as f:
    f.write('sub-001\nsub-002\nsub-003\n')

print(cltpipe.get_fullids2process(str(Path(tempfile.gettempdir()) / 'all_t1s_ids.txt')))


---
## <a id="pipe-inventory"></a>2. Inventory of the Derivatives
Methods to scan the derivatives folders and build the file inventories.

### 2.1 `build_derivatives_inventory`
<a id="pipe-build_derivatives_inventory"></a>

Build a combined file inventory across all the pipeline derivative folders.

In [ ]:
######################## Testing build_derivatives_inventory ########################

import tempfile
from pathlib import Path
import clabtoolkit.pipelinetools as cltpipe
import clabtoolkit.bidstools as cltbids
import itables
import os

# Generating a BIDs dataset
bids_dir = str(Path(tempfile.gettempdir()) / "sim_bids_test2")   # <-- Replace with your BIDS root directory
deriv_dir = os.path.join(bids_dir, 'derivatives') # <-- Replace with your derivatives directory path if different

# ------ Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #
# Create a simulated BIDS dataset for testing purposes 
print(f'Creating a simulated BIDS dataset in {bids_dir}...')

nsub = 3 # Number of subjects
nses = 2 # Number of visits
nruns = 2 # Number of runs per modality
modalities = ["anat"] # Define the modalities to include in the simulated dataset

print(f'Number of subjects: {nsub}')
print(f'Number of visits: {nses}')

cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=nsub,
        n_visits=nses,
        modalities=modalities,
        include_derivatives=['thalseg-cnn', 'chimera', 'fsl-first'],
        runs_per_modality=nruns,
        overwrite=True,
    )

# ---- End of Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #
print('\n Test 1: Building derivatives inventory')
tmp_table = cltpipe.build_derivatives_inventory(deriv_dir)
itables.show(tmp_table)

print('\n Test 2: Building derivatives inventory with specific pipeline directories')
tmp_table = cltpipe.build_derivatives_inventory(deriv_dir, pipe_dirs=['chimera', 'fsl-first'])
itables.show(tmp_table)

print('\n Test 3: Building derivatives inventory with multiple pipeline directories and saving to CSV')
out_table = os.path.join(deriv_dir, 'derivatives_inventory.csv')
tmp_table = cltpipe.build_derivatives_inventory(deriv_dir, pipe_dirs=['chimera', 'fsl-first'], output_table=out_table, overwrite=True)
print(f'Derivatives inventory saved to: {out_table}')

---
## <a id="pipe-status"></a>3. Processing Status of the Pipelines
Methods for assessing the processing status of the pipelines.

### 3.1 `create_processing_status_table`
<a id="pipe-create_processing_status_table"></a>

Creates a table with the processing status of the subjects in the BIDs derivatives directory

In [ ]:
######################## Testing create_processing_status_table ########################

import tempfile
from pathlib import Path
import clabtoolkit.pipelinetools as cltpipe
import clabtoolkit.bidstools as cltbids
import itables
import os
# Generating a BIDs dataset
bids_dir = str(Path(tempfile.gettempdir()) / "sim_bids_test1")   # <-- Replace with your BIDS root directory
deriv_dir = os.path.join(bids_dir, 'derivatives') # <-- Replace with your derivatives directory path if different

# ------ Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #
# Create a simulated BIDS dataset for testing purposes 
print(f'Creating a simulated BIDS dataset in {bids_dir}...')

nsub = 3 # Number of subjects
nses = 2 # Number of visits
nruns = 2 # Number of runs per modality
modalities = ["anat"] # Define the modalities to include in the simulated dataset

print(f'Number of subjects: {nsub}')
print(f'Number of visits: {nses}')

cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=nsub,
        n_visits=nses,
        modalities=modalities,
        include_derivatives=['thalseg-cnn', 'chimera', 'fsl-first'],
        runs_per_modality=nruns,
        overwrite=True,
    )

# ---- End of Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #

# Get the list of Full IDs to process from the BIDs dataset
full_ids = cltpipe.get_fullids2process(in_dir=bids_dir, source='files')

print('\n Test1: Creating processing status table for simulated BIDs dataset')
deriv_table, _ = cltpipe.create_processing_status_table(deriv_dir, full_ids=full_ids)
itables.show(deriv_table)

print('\n Test2: Creating processing status table and save it to a CSV file')
csv_path = os.path.join(deriv_dir, 'processing_status_table.csv')
deriv_table, csv_path = cltpipe.create_processing_status_table(deriv_dir, full_ids=full_ids, output_table=csv_path)
itables.show(deriv_table)
print(f'Processing status table saved to: {csv_path}')



### 3.2 `get_processing_status_details_json`
<a id="pipe-get_processing_status_details_json"></a>

Create a dictionary with the details of the processing status of the subjects and store it as a JSON file.

In [ ]:
######################## Testing get_processing_status_details_json ########################

import tempfile
from pathlib import Path
import clabtoolkit.pipelinetools as cltpipe
import clabtoolkit.bidstools as cltbids
import clabtoolkit.misctools as cltmisc
import os

# Generating a BIDs dataset
bids_dir = str(Path(tempfile.gettempdir()) / "sim_bids_test4")   # <-- Replace with your BIDS root directory
deriv_dir = os.path.join(bids_dir, 'derivatives') # <-- Replace with your derivatives directory path if different

# ------ Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #
# Create a simulated BIDS dataset for testing purposes 
print(f'Creating a simulated BIDS dataset in {bids_dir}...')

nsub = 5 # Number of subjects
nses = 2 # Number of visits
nruns = 1 # Number of runs per modality
modalities = ["anat"] # Define the modalities to include in the simulated dataset

print(f'Number of subjects: {nsub}')
print(f'Number of visits: {nses}')

cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=nsub,
        n_visits=nses,
        modalities=modalities,
        include_derivatives=['thalseg-cnn', 'chimera', 'fsl-first'],
        runs_per_modality=nruns,
        overwrite=True,
    )

# Remove 2 files from the simulated chimera dataset and 1 file from the fsl-first dataset to test processing status
import os
os.remove(os.path.join(deriv_dir, 'chimera', 'sub-01', 'ses-01','anat', 'sub-01_ses-01_T1w.nii.gz'))
os.remove(os.path.join(deriv_dir, 'chimera', 'sub-01', 'ses-02','anat', 'sub-01_ses-02_T1w.nii.gz'))
os.remove(os.path.join(deriv_dir, 'fsl-first', 'sub-05', 'ses-02','anat', 'sub-05_ses-02_T1w.nii.gz'))

# ---- End of Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #

# Get the list of Full IDs to process from the BIDs dataset
full_ids = cltpipe.get_fullids2process(in_dir=bids_dir, source='files')

print('\n Test 1: Creating processing status table for simulated BIDs dataset')
output_table = os.path.join(deriv_dir, 'processing_status_table.tsv')
deriv_table, _ = cltpipe.create_processing_status_table(deriv_dir, full_ids=full_ids, output_table=output_table)

processed_subjects, _ = cltpipe.get_processing_status_details_json(
    proc_status_df=output_table,
    full_ids=full_ids,
    deriv_dir=deriv_dir,
    pipe_dirs=["chimera"], 
    only_ids=True # To get only the full IDs
)

data_new = cltmisc.ExplorerDict(processed_subjects)

# Print updated tree
print('Updated Dictionary Tree:')
print('The Full Ids containing missing files for the selected pipelines are:')
data_new.tree()

print('\n Test 2: Creating processing status table for simulated BIDs dataset')

processed_subjects, _ = cltpipe.get_processing_status_details_json(
    proc_status_df=output_table,
    full_ids=full_ids,
    deriv_dir=deriv_dir,
    pipe_dirs=["chimera", "fsl-first"], 
    only_ids=False # To get detailed information, not just IDs
)

data_new = cltmisc.ExplorerDict(processed_subjects)

# Print updated tree
print('Updated Dictionary Tree:')
print('The Full Ids containing missing files for the selected pipelines are:')
data_new.tree()

### 3.3 `get_processing_status_details_sqlite3`
<a id="pipe-get_processing_status_details_sqlite3"></a>

Create a dictionary with the details of the processing status of the subjects and store it in a SQLite3 database.

In [ ]:
######################## Testing get_processing_status_details_sqlite3 ########################

import tempfile
from pathlib import Path
import clabtoolkit.pipelinetools as cltpipe
import clabtoolkit.bidstools as cltbids
import clabtoolkit.misctools as cltmisc
import os

# Generating a BIDs dataset
bids_dir = str(Path(tempfile.gettempdir()) / "sim_bids_test5")   # <-- Replace with your BIDS root directory
deriv_dir = os.path.join(bids_dir, 'derivatives') # <-- Replace with your derivatives directory path if different

# ------ Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #
# Create a simulated BIDS dataset for testing purposes 
print(f'Creating a simulated BIDS dataset in {bids_dir}...')

nsub = 5 # Number of subjects
nses = 2 # Number of visits
nruns = 1 # Number of runs per modality
modalities = ["anat"] # Define the modalities to include in the simulated dataset

print(f'Number of subjects: {nsub}')
print(f'Number of visits: {nses}')

cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=nsub,
        n_visits=nses,
        modalities=modalities,
        include_derivatives=['thalseg-cnn', 'chimera', 'fsl-first'],
        runs_per_modality=nruns,
        overwrite=True,
    )

# Remove 2 files from the simulated chimera dataset and 1 file from the fsl-first dataset to test processing status
import os
os.remove(os.path.join(deriv_dir, 'chimera', 'sub-01', 'ses-01','anat', 'sub-01_ses-01_T1w.nii.gz'))
os.remove(os.path.join(deriv_dir, 'chimera', 'sub-01', 'ses-02','anat', 'sub-01_ses-02_T1w.nii.gz'))
os.remove(os.path.join(deriv_dir, 'fsl-first', 'sub-05', 'ses-02','anat', 'sub-05_ses-02_T1w.nii.gz'))

# ---- End of Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #

# Get the list of Full IDs to process from the BIDs dataset
full_ids = cltpipe.get_fullids2process(in_dir=bids_dir, source='files')

print('\n Test 1: Creating processing status table for simulated BIDs dataset')
output_table = os.path.join(deriv_dir, 'processing_status_table.tsv')
deriv_table, _ = cltpipe.create_processing_status_table(deriv_dir, full_ids=full_ids, output_table=output_table)

processed_subjects, _ = cltpipe.get_processing_status_details_sqlite3(
    proc_status_df=output_table,
    full_ids=full_ids,
    deriv_dir=deriv_dir,
    pipe_dirs=["chimera"], 
    only_ids=True # To get only the full IDs
)

data_new = cltmisc.ExplorerDict(processed_subjects)

# Print updated tree
print('Updated Dictionary Tree:')
print('The Full Ids containing missing files for the selected pipelines are:')
data_new.tree()

print('\n Test 2: Creating processing status table for simulated BIDs dataset and saving to SQLite3 database')
proc_db = os.path.join(deriv_dir, "all_subjects_processing_status_details.db")
processed_subjects, _ = cltpipe.get_processing_status_details_sqlite3(
    proc_status_df=output_table,
    full_ids=full_ids,
    deriv_dir=deriv_dir,
    pipe_dirs=["chimera", "fsl-first"], 
    only_ids=False, # To get detailed information, not just IDs
    db_path=proc_db
)

data_new = cltmisc.ExplorerDict(processed_subjects)

# Print updated tree
print('Updated Dictionary Tree:')
print('The Full Ids containing missing files for the selected pipelines are:')
data_new.tree()

### 3.4 `query_processing_status_db`
<a id="pipe-query_processing_status_db"></a>

Query the processing status database to extract useful information.

In [ ]:
######################## Testing query_processing_status_db ########################

import tempfile
from pathlib import Path
import clabtoolkit.pipelinetools as cltpipe
import clabtoolkit.bidstools as cltbids
import itables
import os

# Generating a BIDs dataset
bids_dir = str(Path(tempfile.gettempdir()) / "sim_bids_test5")   # <-- Replace with your BIDS root directory
deriv_dir = os.path.join(bids_dir, 'derivatives') # <-- Replace with your derivatives directory path if different

# ------ Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #
# Create a simulated BIDS dataset for testing purposes 
print(f'Creating a simulated BIDS dataset in {bids_dir}...')

nsub = 5 # Number of subjects
nses = 2 # Number of visits
nruns = 1 # Number of runs per modality
modalities = ["anat"] # Define the modalities to include in the simulated dataset

print(f'Number of subjects: {nsub}')
print(f'Number of visits: {nses}')

cltbids.create_a_simulated_bids_dataset(
        output_dir=bids_dir,
        n_subjects=nsub,
        n_visits=nses,
        modalities=modalities,
        include_derivatives=['thalseg-cnn', 'chimera', 'fsl-first'],
        runs_per_modality=nruns,
        overwrite=True,
    )

# Remove 2 files from the simulated chimera dataset and 1 file from the fsl-first dataset to test processing status
import os
os.remove(os.path.join(deriv_dir, 'chimera', 'sub-01', 'ses-01','anat', 'sub-01_ses-01_T1w.nii.gz'))
os.remove(os.path.join(deriv_dir, 'chimera', 'sub-01', 'ses-02','anat', 'sub-01_ses-02_T1w.nii.gz'))
os.remove(os.path.join(deriv_dir, 'fsl-first', 'sub-05', 'ses-02','anat', 'sub-05_ses-02_T1w.nii.gz'))

# ---- End of Simulated BIDS dataset creation (REMOVE this section if using a real dataset) ----- #

# Get the list of Full IDs to process from the BIDs dataset
full_ids = cltpipe.get_fullids2process(in_dir=bids_dir, source='files')

output_table = os.path.join(deriv_dir, 'processing_status_table.tsv')
deriv_table, _ = cltpipe.create_processing_status_table(deriv_dir, full_ids=full_ids, output_table=output_table)

print('\n Test 2: Creating processing status table for simulated BIDs dataset and saving to SQLite3 database')
proc_db = os.path.join(deriv_dir, "all_subjects_processing_status_details.db") 
print(f"Input: proc_status_df={proc_db}")
processed_subjects, _ = cltpipe.get_processing_status_details_sqlite3(
    proc_status_df=output_table,
    full_ids=full_ids,
    deriv_dir=deriv_dir,
    pipe_dirs=["chimera", "fsl-first"], 
    only_ids=False, # To get detailed information, not just IDs
    db_path=proc_db
)

# Get all subjects with mismatches
subjects_df = cltpipe.query_processing_status_db(proc_db, "subjects_with_mismatches")
print(f"Found {len(subjects_df)} subjects with mismatches")
itables.show(subjects_df)
print(" ")

# Get pipeline statistics
# Test 2: Querying the processing status database for pipeline statistics
print('\nTest 2: Querying the processing status database for pipeline statistics')
pipeline_stats = cltpipe.query_processing_status_db(proc_db, "pipelines_with_mismatches")
print(f"Found {len(pipeline_stats)} pipelines with mismatches")
print("Pipelines with the most mismatches:")
print(pipeline_stats)
print(" ")

# Test 3: Querying the processing status database for a specific pipeline
print('\nTest 3: Querying the processing status database for a specific pipeline')
pipeline = "thalseg-cnn"
freesurfer_missing = cltpipe.query_processing_status_db(
    proc_db, 
    "missing_files_count", 
    pipeline=pipeline
)
print(f"Found {len(freesurfer_missing)} subjects with missing FreeSurfer files")
print(" ")


